In [52]:
parameters = {
    'n_paths': ['planner_stats', 'n_paths'],
    'len_m': ['planner_stats', 'len_m'],
    'yaw_changes': ['planner_stats', 'yaw_changes'],
    'coverage_p': ['zone_stats', 'coverage_p'],
    'overlap_p': ['zone_stats', 'overlap_p']
}

In [53]:
import os
import json
import pandas as pd
from pathlib import Path

In [54]:
def collect_results(parameter_path):
    """Collect all result.json files and extract relevant data."""
    data = []

    # Find all result.json files recursively
    for json_file in Path('.').rglob('result.json'):
        try:
            with open(json_file, 'r') as f:
                result = json.load(f)

            # Extract required fields
            map_name = result.get('map', 'Unknown')
            method = result.get('method', 'Unknown')

            # Navigate through nested dictionary using parameter_path
            value = result
            for key in parameter_path:
                value = value.get(key, {})

            # If final value is not a dict, it's our target value
            if not isinstance(value, dict) and value is not None:
                data.append({
                    'map': map_name,
                    'method': method,
                    'value': value
                })

        except (json.JSONDecodeError, FileNotFoundError, KeyError) as e:
            print(f"Error reading {json_file}: {e}")

    return data

In [55]:
def create_table(data):
    """Create a pivot table with methods as rows and maps as columns."""
    if not data:
        print("No valid data found.")
        return None

    # Convert to DataFrame
    df = pd.DataFrame(data)

    # Create pivot table
    table = df.pivot_table(
        index='method',
        columns='map',
        values='value',
        aggfunc='first'  # In case of duplicates, take first value
    )

    return table

In [56]:
def get_table(param):
    parameter_path = parameters.get(param, parameters['len_m'])
    data = collect_results(parameter_path)
    table = create_table(data)
    return table

In [57]:
get_table("n_paths")

map,Map0,Map1,Map2,Map4,map_fr52
method,,,,,
m2_boust_s3_milp,1,17,10,15,24
m3_contour_s3_milp,1,8,3,6,17


# len_m

In [58]:
get_table("len_m")

map,Map0,Map1,Map2,Map4,map_fr52
method,,,,,
m2_boust_s3_milp,121.72,151.99,108.15,122.21,452.68
m3_contour_s3_milp,126.67,158.35,86.30,115.12,448.56


# yaw_changes

In [59]:
get_table("yaw_changes")

map,Map0,Map1,Map2,Map4,map_fr52
method,,,,,
m2_boust_s3_milp,34.56,152.87,85.15,95.05,247.16
m3_contour_s3_milp,37.73,159.44,74.14,133.10,356.32


# coverage_p

In [60]:
get_table("coverage_p")

map,Map0,Map1,Map2,Map4,map_fr52
method,,,,,
m2_boust_s3_milp,101.94,85.19,91.93,86.54,87.63
m3_contour_s3_milp,100.49,93.69,95.36,91.54,96.48


# overlap_p

In [61]:
get_table("overlap_p")

map,Map0,Map1,Map2,Map4,map_fr52
method,,,,,
m2_boust_s3_milp,20.13,53.38,56.59,46.71,46.53
m3_contour_s3_milp,26.53,50.68,23.15,33.99,36.46
